# expiMap on cancer data (Colab)

In [ ]:
# 1. connect Google Drive
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/expimap_cancer'
import os; os.makedirs(DRIVE, exist_ok=True); print('results go to', DRIVE)

In [ ]:
# 2. check the GPU
!nvidia-smi | head -n 12
import torch; print(torch.__version__, torch.cuda.is_available())

In [ ]:
# 3. get the code
REPO = 'https://github.com/Hastikabirsamadi/expiMap-cancer-pathways.git'
%cd /content
!git clone $REPO repo || git -C repo pull
%cd repo

In [ ]:
# 4. install packages and get the PBMC reference data and Reactome pathways
%pip install -q -r requirements-colab.txt
!python download_data.py

## CTCL (blood cancer)
Downloads the CTCL atlas blood file, keeps PBMC samples only (at most 1000 cells per donor), and saves the small subsample to Drive. After the first run, the download and subsampling are skipped.

In [ ]:
# 5. downloads the full file to Colab's local disk, then subsample to Drive
import os
URL = 'https://datasets.cellxgene.cziscience.com/ca954737-f273-4144-b29d-1dccdbd22e4c.h5ad'
FULL = '/content/ctcl_blood.h5ad'
SUB = f'{DRIVE}/ctcl_pbmc_subsample.h5ad'
if not os.path.exists(SUB):
    !wget -c -q --show-progress -O $FULL $URL
    !python prepare_ctcl.py --h5ad $FULL --out $SUB --per_donor 1000
else:
    print('subsample already on Drive:', SUB)

In [ ]:
# 6. map CTCL onto the PBMC reference by surgery (about the time of one baseline run)
# REF_MODEL: the trained PBMC reference from the baseline project (unmodified expiMap, seed 0), read straight from Drive.
# If this folder had no model, a new one would be trained and saved there.
REF_MODEL = '/content/drive/MyDrive/expimap_results/runs/baseline/seed0/ref_model'
OUT = f'{DRIVE}/ctcl_map/seed0'
!python map_ctcl.py --ctcl $SUB --ref_model $REF_MODEL --out $OUT --seed 0

In [ ]:
# 7. look at the results
# the figures are redrawn from the saved results (no training), so this also updates an older run
OUT = f'{DRIVE}/ctcl_map/seed0'
!python plot_ctcl.py --out $OUT
import json, pandas as pd
from IPython.display import Image, display
print(json.dumps(json.load(open(f'{OUT}/metrics.json')), indent=1))
for f in ['fig1_cell_types', 'fig2_disease', 'fig3_author_labels', 'fig4_confidence', 'fig5_mixing']:
    display(Image(f'{OUT}/{f}.png'))
display(pd.read_csv(f'{OUT}/crosstab_normal.csv', index_col=0))

## Part A: which pathways change in the cancer cells
Uses the saved mapping (no training, no GPU needed). Compares malignant with non-malignant CD4+ T cells inside each patient, and MF with Sezary inside the studies that have both. Results go to `ctcl_map/seed0/partA`.

In [ ]:
# 8. Part A
OUT = f'{DRIVE}/ctcl_map/seed0'
!python pathways_ctcl.py --map $OUT --counts {DRIVE}/ctcl_pbmc_subsample.h5ad
import pandas as pd
from IPython.display import Image, display
A = f'{OUT}/partA'
display(pd.read_csv(f'{A}/comp1_malignant_vs_cd4.csv').head(20))
for f in ['fig_A1_malignant_vs_cd4_heatmap', 'fig_A2_pathway_umaps', 'fig_A3_celltype_profiles', 'fig_A4_mf_vs_sezary', 'fig_A5_gene_check']:
    try: display(Image(f'{A}/{f}.png'))
    except FileNotFoundError: print('not made:', f)

## Seed check: do the Part A results hold up with other random seeds?
Maps CTCL again with the baseline reference models of seeds 1 and 2 (needs the GPU, about 10 minutes per seed), runs Part A on each, then compares all three seeds. Finished seeds are skipped on a rerun.

In [ ]:
# 9. seed check 
import os
SUB = f'{DRIVE}/ctcl_pbmc_subsample.h5ad'
for s in [1, 2]:
    REF = f'/content/drive/MyDrive/expimap_results/runs/baseline/seed{s}/ref_model'
    if not os.path.exists(f'{REF}/model_params.pt'):
        raise SystemExit(f'no baseline reference model for seed {s} at {REF}; stop (do not train into the baseline folder)')
    O = f'{DRIVE}/ctcl_map/seed{s}'
    if not os.path.exists(f'{O}/metrics.json'):
        !python map_ctcl.py --ctcl $SUB --ref_model $REF --out $O --seed $s
    if not os.path.exists(f'{O}/partA/summary.json'):
        !python pathways_ctcl.py --map $O --counts $SUB
R = f'{DRIVE}/ctcl_map'
!python compare_seeds.py --runs {R}/seed0 {R}/seed1 {R}/seed2 --out {R}/seed_comparison
from IPython.display import Image, display
display(Image(f'{R}/seed_comparison/fig_seed_agreement.png'))

# Part B: model understanding
## Are pathway scores already different between seeds on healthy cells?
The three baseline reference models (seeds 0, 1, 2) score the same healthy PBMC cells. No cancer data, no training, no GPU needed. Results go to `partB0` (the folder keeps its first name).

In [ ]:
# 10. Part B 
RUNS = '/content/drive/MyDrive/expimap_results/runs/baseline'
B0 = f'{DRIVE}/partB0'
!python partb_ref_seeds.py --refs {RUNS}/seed0/ref_model {RUNS}/seed1/ref_model {RUNS}/seed2/ref_model --out $B0
from IPython.display import Image, display
display(Image(f'{B0}/fig_B0_agreement.png')); display(Image(f'{B0}/fig_B0_contrasts.png'))

# Part C: limiting the pathway gene weights in scArches
Trains the healthy reference again with two limited versions of the decoder (`positive`: gene weights can only be positive, `fixed`: all genes of a pathway share one weight), 3 seeds each, and compares them with the original (`free`, your baseline runs) on stability and mapping quality. GPU runtime, about 15 minutes per training (6 trainings). Finished runs are skipped on a rerun.

In [ ]:
# 11. Part C (GPU runtime; needs cells 1, 3 and 4 after a disconnect)
import os
FIX = f'{DRIVE}/fix_runs'
for v in ['positive', 'fixed']:
    for s in [0, 1, 2]:
        if not os.path.exists(f'{FIX}/{v}/seed{s}/metrics.json'):
            !python run_experiment.py --variant $v --seed $s --out $FIX --save_models
BASE = '/content/drive/MyDrive/expimap_results/runs/baseline'
!python compare_fix.py --runs free={BASE} positive={FIX}/positive fixed={FIX}/fixed --out {DRIVE}/partC
from IPython.display import Image, display
display(Image(f'{DRIVE}/partC/fig_fix_stability.png')); display(Image(f'{DRIVE}/partC/fig_fix_mapping.png'))

## Part C: does pathway activity move between pathways that share genes?
No training, no GPU needed (about a few minutes). Every reference model scores the same healthy cells again, and the script checks whether another seed holds the same information split over different, overlapping pathways.

In [ ]:
# 12. Part C overlap test (CPU is fine; needs cells 1, 3 and 4 after a disconnect)
BASE = '/content/drive/MyDrive/expimap_results/runs/baseline'
FIX = f'{DRIVE}/fix_runs'
!python partc_overlap.py --runs free={BASE} positive={FIX}/positive fixed={FIX}/fixed --out {DRIVE}/partC_overlap
from IPython.display import Image, display
display(Image(f'{DRIVE}/partC_overlap/fig_C_overlap_r2.png')); display(Image(f'{DRIVE}/partC_overlap/fig_C_overlap_scatter.png'))

## Part C: what makes the few stable pathways different?
No training, no GPU needed. Compares each pathway's stability between seeds with how much information it carries, how much the decoder uses it, and how well it follows cell types and the main directions of the data.

In [ ]:
# 13. Part C diagnosis (CPU is fine; needs cells 1, 3 and 4 after a disconnect)
BASE = '/content/drive/MyDrive/expimap_results/runs/baseline'
FIX = f'{DRIVE}/fix_runs'
!python partc_diagnose.py --runs free={BASE} positive={FIX}/positive fixed={FIX}/fixed --out {DRIVE}/partC_diagnose
from IPython.display import Image, display
display(Image(f'{DRIVE}/partC_diagnose/fig_C_diagnose.png'))

## Part C: living pathways only
No training, no GPU needed. Counts the 'dead' pathways (the encoder says the same for every cell), checks the seed agreement on the living ones, and repeats the Part A seed check on the CTCL cells with living pathways only.

In [ ]:
# 14. Part C living pathways (CPU is fine; needs cells 1, 3 and 4 after a disconnect)
BASE = '/content/drive/MyDrive/expimap_results/runs/baseline'
FIX = f'{DRIVE}/fix_runs'
AL = f'{DRIVE}/partC_alive'
!python partc_alive.py --runs free={BASE} positive={FIX}/positive fixed={FIX}/fixed --out {AL}
# Part A seed check again, living pathways only (the CTCL maps used the baseline = free reference models)
R = f'{DRIVE}/ctcl_map'
!python compare_seeds.py --runs {R}/seed0 {R}/seed1 {R}/seed2 --out {R}/seed_comparison_alive --keep {AL}/alive_free.csv
from IPython.display import Image, display
display(Image(f'{AL}/fig_C_alive.png')); display(Image(f'{R}/seed_comparison_alive/fig_seed_agreement.png'))